<a href="https://colab.research.google.com/github/darelphilipo/indian-subreddit-automod-kit/blob/main/hinglish_hatespeech.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title
!pip install -q duckdb pandas tqdm huggingface_hub

import duckdb
import pandas as pd
import os
from huggingface_hub import HfApi
from tqdm.notebook import tqdm
from google.colab import drive

# =====================================================================
# 1. GOOGLE DRIVE PERSISTENCE SETUP
# =====================================================================
print("🔗 Checking Google Drive connection...")
drive.mount('/content/drive', force_remount=False)

PROJECT_DIR = '/content/drive/MyDrive/Hinglish_NLP_Project/Production'
os.makedirs(PROJECT_DIR, exist_ok=True)
print(f"✅ Safe storage locked: {PROJECT_DIR}\n")

# =====================================================================
# 2. SUBREDDIT TAXONOMY & STRATIFIED QUOTAS (~35,000 Target Rows)
# =====================================================================
SUBREDDIT_TAXONOMY = {
    "Cat1_Main_India": {
        "quota_per_sub": 1500,
        "subs": ["india", "IndiaSpeaks", "indiasocial", "indiadiscussion", "unitedstatesofindia", "bakchodi", "librandu", "indianews"]
    },
    "Cat2_States_Cities": {
        "quota_per_sub": 400,
        "subs": ["delhi", "bangalore", "mumbai", "kolkata", "Chennai", "hyderabad", "pune", "Kerala", "TamilNadu", "bihar", "Maharashtra", "uttarpradesh", "Kashmiri", "Haryana"]
    },
    "Cat3_Culture_Entertainment": {
        "quota_per_sub": 500,
        "subs": ["bollywood", "BollyBlindsNGossip", "tollywood", "kollywood", "MalayalamMovies", "IndianFood", "IndianHipHopHeads", "indianpeoplefacebook"]
    },
    "Cat4_Sports_Gaming_Finance": {
        "quota_per_sub": 600,
        "subs": ["Cricket", "CricketShitpost", "ipl", "IndianGaming", "indianbikes", "CarsIndia", "IndianStreetBets", "IndianStockMarket"]
    },
    "Cat5_Diaspora_Desi": {
        "quota_per_sub": 800,
        "subs": ["ABCDesis", "DesiMeta", "SouthAsianMasculinity"]
    },
    "Cat6_Academia_Social_Memes": {
        "quota_per_sub": 600,
        "subs": ["IndianTeenagers", "developersIndia", "Indian_Academia", "LegalAdviceIndia", "TwoXIndia", "indianboysontinder", "indiangirlsontinder", "IndianDankMemes", "indiameme", "dankinindia"]
    },
    "Cat7_Religion": {
        "quota_per_sub": 800,
        "subs": ["hinduism", "atheismindia", "indianmuslims", "Chodi", "Sham_Sharma_Show", "Sikh", "EXHINDU"]
    }
}

# =====================================================================
# 3. DYNAMIC CLOUD SHARD DISCOVERY (Zero-Guessing Engine)
# =====================================================================
print("🕵️ Locating active comment shards in Hugging Face archive...")
api = HfApi()

try:
    all_files = api.list_repo_files("open-index/arctic", repo_type="dataset")
    # Strictly isolate comment databases (preventing submission/post schema errors)
    comment_files = [f for f in all_files if f.endswith('.parquet') and 'data/comments/' in f]

    # Sort chronologically descending to pull the newest indexed data first
    comment_files.sort(reverse=True)

    # Target 300 shards (~15 million global comments) for deep regional coverage
    target_shards = comment_files[:300]
    hf_urls = [f"hf://datasets/open-index/arctic/{f}" for f in target_shards]

    print(f"✅ Discovered {len(comment_files):,} total comment archives.")
    print(f"⚡ Network engine locked onto 300 newest shards starting from: '{target_shards[0]}'\n")
except Exception as e:
    raise RuntimeError(f"❌ Failed to inspect Hugging Face repository: {e}")

# Initialize DuckDB out-of-core cloud streaming
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")

# =====================================================================
# 4. CATEGORY-BY-CATEGORY STREAMING & SANITATION ENGINE
# =====================================================================
master_dataset = []
print("🚀 STARTING PRODUCTION EXTRACTION...\n")

for cat_name, cat_data in SUBREDDIT_TAXONOMY.items():
    quota = cat_data["quota_per_sub"]
    subs_list = [s.lower() for s in cat_data["subs"]]

    # Format subreddits for SQL IN clause: ('india', 'indiaspeaks', ...)
    sql_subs = ", ".join([f"'{s}'" for s in subs_list])

    print(f"⏳ Streaming {cat_name} ({len(subs_list)} subreddits | Target: {quota} rows/sub)...")

    # Batch query executes predicate pushdown across all 300 shards simultaneously
    query = f"""
    SELECT
        LOWER(subreddit) AS subreddit_clean,
        author,
        score,
        body AS text,
        created_at,
        -- Pre-allocating the 6-Pillar Multi-Label Schema initialized to 0
        0 AS casteist,
        0 AS communal_religious,
        0 AS xenophobic_regional,
        0 AS misogynistic_sexual,
        0 AS abusive_profanity,
        0 AS passive_aggressive
    FROM read_parquet({hf_urls})
    WHERE LOWER(subreddit) IN ({sql_subs})

      -- AGGRESSIVE SANITATION & ANTI-SPAM FILTERS --
      AND body NOT IN ('[deleted]', '[removed]', '', '[removed by reddit]')
      AND body IS NOT NULL
      AND LOWER(author) NOT IN ('automoderator', 'bot', 'deleted', 'reddit', 'transcribot')

      -- Eliminate SEO link farms and Markdown URL structures: [Text](url)
      AND body NOT LIKE '%http%'
      AND body NOT LIKE '%www.%'
      AND body NOT LIKE '%](%'

      -- Enforce human conversational lengths (drops copy-paste walls & 1-word spam)
      AND length(body) BETWEEN 30 AND 800

      -- Demand community validation (drops unvoted automated spam)
      AND score >= 2

    ORDER BY score DESC;
    """

    try:
        df_cat = con.query(query).to_df()

        if not df_cat.empty:
            # Enforce exact per-subreddit quota via Python stratified sampling
            df_stratified = df_cat.groupby('subreddit_clean').head(quota).copy()

            # Map clean lowercase names back to original subreddit capitalization
            sub_map = {s.lower(): s for s in cat_data["subs"]}
            df_stratified['subreddit'] = df_stratified['subreddit_clean'].map(sub_map)
            df_stratified.drop(columns=['subreddit_clean'], inplace=True)

            master_dataset.append(df_stratified)
            print(f"   ✅ Extracted {len(df_stratified):,} clean rows across {df_stratified['subreddit'].nunique()} subreddits.")
        else:
            print(f"   ⚠️ No matching clean rows found for {cat_name} in these 300 shards.")

    except Exception as e:
        print(f"   ❌ SQL Stream Error on {cat_name}: {e}")

# =====================================================================
# 5. DATASET MERGING, SHUFFLING & GOOGLE DRIVE EXPORT
# =====================================================================
if master_dataset:
    print("\n📦 Merging categories and applying global shuffle...")
    df_final = pd.concat(master_dataset, ignore_index=True)

    # Shuffle dataset so MuRIL sees an evenly distributed mix of slang and categories
    df_final = df_final.sample(frac=1, random_state=42).reset_index(drop=True)

    # Save master file (with metadata intact for internal auditing)
    output_path = os.path.join(PROJECT_DIR, "production_hinglish_master_unlabeled.csv")
    df_final.to_csv(output_path, index=False)

    print("\n" + "="*65)
    print(f"🎉 PRODUCTION EXTRACTION COMPLETE!")
    print(f"📊 Total Clean Rows Extracted : {len(df_final):,}")
    print(f"📁 Master Dataset Saved To     : {output_path}")
    print("="*65)

    # Display subreddit distribution preview
    print("\n📈 Top 15 Subreddits by Extracted Volume:")
    print(df_final['subreddit'].value_counts().head(15))

    print("\n📋 Dataset Structure Preview:")
    display(df_final[['subreddit', 'score', 'text', 'casteist', 'abusive_profanity']].head(10))
else:
    print("\n❌ Extraction failed. Please verify your internet connection and check error logs.")

🔗 Checking Google Drive connection...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Safe storage locked: /content/drive/MyDrive/Hinglish_NLP_Project/Production

🕵️ Locating active comment shards in Hugging Face archive...
✅ Discovered 19,133 total comment archives.
⚡ Network engine locked onto 300 newest shards starting from: 'data/comments/2021/04/409.parquet'

🚀 STARTING PRODUCTION EXTRACTION...

⏳ Streaming Cat1_Main_India (8 subreddits | Target: 1500 rows/sub)...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   ✅ Extracted 10,938 clean rows across 8 subreddits.
⏳ Streaming Cat2_States_Cities (14 subreddits | Target: 400 rows/sub)...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   ✅ Extracted 3,837 clean rows across 13 subreddits.
⏳ Streaming Cat3_Culture_Entertainment (8 subreddits | Target: 500 rows/sub)...


...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   ✅ Extracted 5,296 clean rows across 10 subreddits.
⏳ Streaming Cat7_Religion (7 subreddits | Target: 800 rows/sub)...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   ✅ Extracted 4,445 clean rows across 7 subreddits.

📦 Merging categories and applying global shuffle...

🎉 PRODUCTION EXTRACTION COMPLETE!
📊 Total Clean Rows Extracted : 33,930
📁 Master Dataset Saved To     : /content/drive/MyDrive/Hinglish_NLP_Project/Production/production_hinglish_master_unlabeled.csv

📈 Top 15 Subreddits by Extracted Volume:
subreddit
india                  1500
librandu               1500
unitedstatesofindia    1500
IndiaSpeaks            1500
indiasocial            1500
indianews              1500
indiadiscussion        1060
bakchodi                878
atheismindia            800
Sham_Sharma_Show        800
Sikh                    800
Chodi                   800
ABCDesis                800
DesiMeta                800
hinduism                800
Name: count, dtype: int64

📋 Dataset Structure Preview:


,subreddit,score,text,casteist,abusive_profanity
0,india,175,Sadly it’s not sarcastic but the harsh reality!,0,0
1,Indian_Academia,8,The physics departments in IITs will be alrigh...,0,0
2,EXHINDU,8,"Ohh good, so he is one of those migrants in ou...",0,0
3,Sham_Sharma_Show,2,It is a close battle and only fools believe in...,0,0
4,TwoXIndia,10,In this regard luckily my mother didn’t have t...,0,0
5,librandu,52,So killing thousands is just a proud Kumbhi tr...,0,0
6,librandu,120,its actually a genius strategy. when you're wh...,0,0
7,developersIndia,4,"Angelist, was where I found mine.\n\nAlso note...",0,0
8,developersIndia,3,Yeah I am talking from experience. Don't build...,0,0
9,CricketShitpost,53,Nostalgic.. nowadays ipl doesn't have they vibe.,0,0


In [ ]:
# @title
# 1. INSTALL THE MODERN SDK
!pip install -q -U google-genai

from google import genai
from google.genai import types
import pandas as pd
import json
import time
import os
import getpass
import re
from google.colab import userdata, drive
from tqdm.auto import tqdm

# =====================================================================
# 2. MOUNT DRIVE & AUTHENTICATE
# =====================================================================
print("🔗 Mounting Google Drive...")
drive.mount('/content/drive', force_remount=False)

print("🔐 Authenticating Gemini API...")
try:
    api_key = userdata.get('GEMINI_API_KEY2')
except Exception as e:
    api_key = getpass.getpass("🔑 Please paste your GEMINI_API_KEY here and hit Enter: ")

client = genai.Client(api_key=api_key)
print("✅ Modern google-genai Client successfully initialized!")

# =====================================================================
# 3. RESUMABLE CHECKPOINT LOGIC
# =====================================================================
PROJECT_DIR = '/content/drive/MyDrive/Hinglish_NLP_Project/Production'
checkpoint_path = os.path.join(PROJECT_DIR, 'gemini_labeled_checkpoint.csv')
input_path = os.path.join(PROJECT_DIR, 'production_hinglish_master_unlabeled.csv')

if os.path.exists(checkpoint_path):
    print(f"📂 Found existing checkpoint. Resuming progress from: {checkpoint_path}")
    df = pd.read_csv(checkpoint_path)
else:
    print(f"📂 Loading fresh unlabelled dataset from: {input_path}")
    df = pd.read_csv(input_path)

if 'gemini_processed' not in df.columns:
    df['gemini_processed'] = 0

categories = [
    'casteist', 'communal_religious', 'xenophobic_regional',
    'misogynistic_sexual', 'abusive_profanity', 'passive_aggressive'
]
for cat in categories:
    if cat not in df.columns:
        df[cat] = 0

# =====================================================================
# 4. COMPRESSED PROMPT CONFIGURATION
# =====================================================================
system_prompt = """You are an expert clinical data annotator for Indian social media and Hinglish content.
Analyze the provided batch of comments. Each comment is preceded by its row ID.

Return ONLY a valid JSON array of objects. To maximize speed, each object MUST use the compact format:
{"id": integer, "f": [c0, c1, c2, c3, c4, c5]}

Where "f" is a 6-element array of binary flags (0 or 1) strictly in this order:
0. casteist (caste slurs, varna discrimination, e.g., chamar, bhangi)
1. communal_religious (religious hostility/dog-whistles, e.g., katwa, sanghi, bhakt, mullah)
2. xenophobic_regional (state/linguistic insults, e.g., bihari, madrasi, bimaru)
3. misogynistic_sexual (gender abuse, harassment, queerphobia, e.g., r#ndi, chakka, hijra)
4. abusive_profanity (standard swear words, trash-talk, e.g., bc, mc, bsdk, chutiya, m4darch0d)
5. passive_aggressive (sarcasm, veiled dog-whistles without explicit swear words)

Example output: [{"id": 0, "f": [0, 1, 0, 0, 0, 1]}, {"id": 1, "f": [0, 0, 0, 0, 1, 0]}]
"""

config = types.GenerateContentConfig(
    system_instruction=system_prompt,
    response_mime_type="application/json",
    temperature=0.0,
    safety_settings=[
        types.SafetySetting(category=types.HarmCategory.HARM_CATEGORY_HATE_SPEECH, threshold=types.HarmBlockThreshold.BLOCK_NONE),
        types.SafetySetting(category=types.HarmCategory.HARM_CATEGORY_HARASSMENT, threshold=types.HarmBlockThreshold.BLOCK_NONE),
        types.SafetySetting(category=types.HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT, threshold=types.HarmBlockThreshold.BLOCK_NONE),
        types.SafetySetting(category=types.HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT, threshold=types.HarmBlockThreshold.BLOCK_NONE),
    ]
)

# =====================================================================
# 5. ULTRA-RESILIENT PARSER UTILITY WITH REGEX FALLBACK
# =====================================================================
def parse_model_output(raw_text):
    """Parses text using native json, falling back to a line-by-line regex if structure is broken."""
    if not raw_text:
        return []

    text_clean = str(raw_text).strip()

    # Strip markdown wrappers if present
    text_clean = re.sub(r'\x60\x60\x60(?:json)?', '', text_clean)
    text_clean = re.sub(r'\x60\x60\x60', '', text_clean)
    text_clean = text_clean.strip()

    # Attempt 1: Direct structural parse
    try:
        return json.loads(text_clean)
    except Exception:
        pass

    # Attempt 2: Extract JSON Array from surrounding text
    json_match = re.search(r'\[\s*\{.*?\}\s*\]', text_clean, re.DOTALL)
    if json_match:
        try:
            return json.loads(json_match.group(0))
        except Exception:
            pass

    # Attempt 3: Regex fallback to isolate valid objects line by line
    parsed_objects = []
    # Matches: id: 123 (or "id": "123") and f: [0, 1, 0, 0, 0, 0]
    pattern = r'["\']?id["\']?\s*:\s*["\']?(\d+)["\']?.*?["\']?f["\']?\s*:\s*\[(.*?)\]'
    matches = re.findall(pattern, text_clean, re.IGNORECASE | re.DOTALL)

    for match in matches:
        try:
            row_id = int(match[0])
            flags_str = match[1]
            flags = [int(x) for x in re.findall(r'[01]', flags_str)]
            if len(flags) >= 6:
                parsed_objects.append({"id": row_id, "f": flags[:6]})
        except Exception:
            continue

    return parsed_objects

# =====================================================================
# 6. HIGH-RELIABILITY SEQUENTIAL PIPELINE (BATCH SIZE = 40)
# =====================================================================
batch_size = 40
unprocessed_indices = df[df['gemini_processed'] == 0].index.tolist()
batches = [unprocessed_indices[i:i + batch_size] for i in range(0, len(unprocessed_indices), batch_size)]

print(f"🚀 Found {len(unprocessed_indices):,} rows remaining to label.")
print(f"📊 Processing {len(batches)} batches sequentially using Gemini 3.1 Flash Lite...")
print(f"🔒 Running a strict 5.2s pacing cycle to protect your 15 RPM free ceiling.\n")

for batch_idx, batch in enumerate(tqdm(batches, desc="Labeling Batches")):
    start_time = time.time()

    # Payload Construction with advanced string isolation
    payload = ""
    for idx in batch:
        clean_text = str(df.at[idx, 'text']).replace('\n', ' ').replace('\r', ' ').replace('"', "'").replace('\\', '').strip()
        payload += f'ID: {idx} | Text: "{clean_text}"\n'

    success = False
    max_attempts = 3 # Reduced to fail faster on hard API blocks
    for attempt in range(max_attempts):
        try:
            response = client.models.generate_content(
                model="gemini-3.1-flash-lite",
                contents=payload,
                config=config
            )

            # Safely extract text in case of a hard API safety refusal
            try:
                raw_text = response.text
            except ValueError:
                raw_text = ""

            results = parse_model_output(raw_text)

            if not results:
                # Capture the model's exact text to see if it refused
                preview = str(raw_text)[:150].replace('\n', ' ') if raw_text else "[BLOCKED BY API SAFETY FILTERS]"
                raise ValueError(f"No valid JSON found. Model said: {preview}")

            # Populate dataframe matrix safely inside the single loop
            for res in results:
                row_id = res.get('id')
                flags = res.get('f')

                if row_id is not None and row_id in df.index and isinstance(flags, list) and len(flags) >= 6:
                    for idx_cat, cat_name in enumerate(categories):
                        df.at[row_id, cat_name] = int(flags[idx_cat])
                    df.at[row_id, 'gemini_processed'] = 1

            success = True
            break

        except Exception as e:
            error_str = str(e)
            if "429" in error_str or "RESOURCE_EXHAUSTED" in error_str:
                print(f"\n⏳ Free daily/minute quota cap triggered at ID {batch[0]}. Cooling down execution for 50 seconds...")
                time.sleep(50)
            else:
                print(f"\n⚠️ Issue at ID {batch[0]} (Attempt {attempt+1}/{max_attempts}). {error_str}")
                time.sleep(3)

    if not success:
        print(f"\n❌ Skipping batch starting with ID {batch[0]} after {max_attempts} failed attempts.")
        print(f"\n--- 🚨 DEBUG: FAILED BATCH DATA (IDs {batch[0]} to {batch[-1]}) ---")
        print(payload)
        print("-------------------------------------------------------------------\n")

    # Pacing Enforcement Loop Execution
    elapsed_time = time.time() - start_time
    sleep_remaining = 5.2 - elapsed_time
    if sleep_remaining > 0:
        time.sleep(sleep_remaining)

    # Auto-save changes securely every 15 batches (~600 rows)
    if (batch_idx + 1) % 15 == 0:
        df.to_csv(checkpoint_path, index=False)

# =====================================================================
# 7. FINAL DATA EXPORT
# =====================================================================
df.to_csv(checkpoint_path, index=False)

muril_columns = ['text'] + categories
muril_export_path = os.path.join(PROJECT_DIR, 'muril_training_data_gemini.csv')
df[df['gemini_processed'] == 1][muril_columns].to_csv(muril_export_path, index=False)

print("\n" + "="*65)
print("🎉 EXECUTION CYCLE SECURE. ENTIRE GHOST-THREAD POOL WIPED!")
print(f"📁 Master Checkpoint Saved At : {checkpoint_path}")
print(f"🚀 MuRIL Ready Training Data At : {muril_export_path}")
print("="*65)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 19.7 MB/s eta 0:00:00
🔗 Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🔐 Authenticating Gemini API...
✅ Modern google-genai Client successfully initialized!
📂 Found existing checkpoint. Resuming progress from: /content/drive/MyDrive/Hinglish_NLP_Project/Production/gemini_labeled_checkpoint.csv
🚀 Found 10,875 rows remaining to label.
📊 Processing 272 batches sequentially using Gemini 3.1 Flash Lite...
🔒 Running a strict 5.2s pacing cycle to protect your 15 RPM free ceiling.



Labeling Batches:   0%|          | 0/272 [00:00<?, ?it/s]


⚠️ Issue at ID 1215 (Attempt 1/3). No valid JSON found. Model said: [BLOCKED BY API SAFETY FILTERS]

⚠️ Issue at ID 1215 (Attempt 2/3). No valid JSON found. Model said: [BLOCKED BY API SAFETY FILTERS]

⚠️ Issue at ID 1215 (Attempt 3/3). No valid JSON found. Model said: [BLOCKED BY API SAFETY FILTERS]

❌ Skipping batch starting with ID 1215 after 3 failed attempts.

--- 🚨 DEBUG: FAILED BATCH DATA (IDs 1215 to 1254) ---
ID: 1215 | Text: "I鈥檓 sorry, he was shot with a *what*?"
ID: 1216 | Text: "But you will be paid 10 rs per photo and with deshbhakti overdose."
ID: 1217 | Text: "I'm from East London where there's a really big South Asian population. It's true that most of them have ended up getting with desi women anyway lol but I think to an extent it may be because it's more convenient for them to bring home a desi girl vs a white girl."
ID: 1218 | Text: "I don't think Bengal has anything to do with the market's current situation."
ID: 1219 | Text: "Wow, nothing has changed since then.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
# @title
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from transformers.trainer_utils import get_last_checkpoint

# ==========================================
# 1. DEFINE PATHS & GOOGLE DRIVE CHECKPOINTS
# ==========================================
DRIVE_PROJECT_DIR = '/content/drive/MyDrive/Hinglish_NLP_Project/Production'
DATASET_PATH = os.path.join(DRIVE_PROJECT_DIR, 'muril_training_data_gemini.csv')
CHECKPOINT_DIR = os.path.join(DRIVE_PROJECT_DIR, 'checkpoints/muril_v1_weighted')
FINAL_MODEL_DIR = os.path.join(DRIVE_PROJECT_DIR, 'models/muril_v1_final')

print("📂 Loading dataset from Drive...")
df = pd.read_csv(DATASET_PATH)

# Clean text column to prevent tokenizer crashes on NaN/numerical values
df = df.dropna(subset=['text'])
df['text'] = df['text'].astype(str)

categories = [
    'casteist', 'communal_religious', 'xenophobic_regional',
    'misogynistic_sexual', 'abusive_profanity', 'passive_aggressive'
]

# ==========================================
# 2. CALCULATE POSITIVE LOSS WEIGHTS
# ==========================================
num_samples = len(df)
pos_counts = df[categories].sum().values
neg_counts = num_samples - pos_counts

pos_weights = neg_counts / (pos_counts + 1e-5)
pos_weight_tensor = torch.tensor(pos_weights, dtype=torch.float32)

print("\n⚖️ Automatically Calculated Loss Weights (pos_weight):")
for cat, weight in zip(categories, pos_weights):
    print(f"  • {cat:<22}: {weight:.2f}x penalty on error")

df['labels'] = df[categories].values.tolist()
df['labels'] = df['labels'].apply(lambda x: [float(i) for i in x])

hf_dataset = Dataset.from_pandas(df[['text', 'labels']])
split_dataset = hf_dataset.train_test_split(test_size=0.2, seed=42)

# ==========================================
# 3. INITIALIZE MuRIL & TOKENIZER
# ==========================================
print("\n🧠 Loading MuRIL tokenizer and base model...")
model_id = "google/muril-base-cased"

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForSequenceClassification.from_pretrained(
    model_id,
    num_labels=len(categories),
    problem_type="multi_label_classification",
    id2label={idx: label for idx, label in enumerate(categories)},
    label2id={label: idx for idx, label in enumerate(categories)}
)

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

print("⚙️ Tokenizing datasets...")
tokenized_datasets = split_dataset.map(tokenize_function, batched=True)

# ==========================================
# 4. CUSTOM WEIGHTED TRAINER
# ==========================================
class WeightedMultiLabelTrainer(Trainer):
    def __init__(self, pos_weight=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.pos_weight = pos_weight

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        device = logits.device
        pos_weight = self.pos_weight.to(device) if self.pos_weight is not None else None

        loss_fct = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        loss = loss_fct(logits, labels)

        return (loss, outputs) if return_outputs else loss

# ==========================================
# 5. TRAINING ARGUMENTS & AUTO-RESUME LOGIC
# ==========================================
training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    fp16=True,
)

trainer = WeightedMultiLabelTrainer(
    pos_weight=pos_weight_tensor,
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    processing_class=tokenizer,           # <-- FIXED: changed from tokenizer=tokenizer to processing_class=tokenizer
)

# Search for existing checkpoint in Drive
last_checkpoint = get_last_checkpoint(CHECKPOINT_DIR) if os.path.exists(CHECKPOINT_DIR) else None

if last_checkpoint is not None:
    print(f"\n🔄 Found existing checkpoint at: {last_checkpoint}")
    print("🚀 Resuming training from checkpoint...")
    trainer.train(resume_from_checkpoint=last_checkpoint)
else:
    print("\n🚀 Starting fresh training...")
    trainer.train()

# ==========================================
# 6. SAVE FINAL MODEL & TOKENIZER
# ==========================================
print(f"\n💾 Saving final model and tokenizer to Drive: {FINAL_MODEL_DIR}")
trainer.save_model(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)

print("\n" + "="*60)
print("🎉 MuRIL V1 WEIGHTED TRAINING COMPLETE!")
print(f"📁 Checkpoints Location : {CHECKPOINT_DIR}")
print(f"🚀 Final Model Location : {FINAL_MODEL_DIR}")
print("="*60)

📂 Loading dataset from Drive...

⚖️ Automatically Calculated Loss Weights (pos_weight):
  • casteist              : 79.12x penalty on error
  • communal_religious    : 6.56x penalty on error
  • xenophobic_regional   : 51.46x penalty on error
  • misogynistic_sexual   : 30.61x penalty on error
  • abusive_profanity     : 8.17x penalty on error
  • passive_aggressive    : 7.16x penalty on error

🧠 Loading MuRIL tokenizer and base model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google/muril-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params w

⚙️ Tokenizing datasets...


Map:   0%|          | 0/27112 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss
1,0.947130,0.906432
2,0.753297,0.874653
3,0.565528,0.945383



💾 Saving final model and tokenizer to Drive: /content/drive/MyDrive/Hinglish_NLP_Project/Production/models/muril_v1_final


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


🎉 MuRIL V1 WEIGHTED TRAINING COMPLETE!
📁 Checkpoints Location : /content/drive/MyDrive/Hinglish_NLP_Project/Production/checkpoints/muril_v1_weighted
🚀 Final Model Location : /content/drive/MyDrive/Hinglish_NLP_Project/Production/models/muril_v1_final


In [1]:
# @title
import os
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# ==========================================
# 1. LOAD TRAINED V1 MODEL FROM DRIVE
# ==========================================
MODEL_PATH = '/content/drive/MyDrive/Hinglish_NLP_Project/Production/models/muril_v1_final'

print("📂 Loading trained V1 MuRIL model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)

# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

categories = [
    'casteist', 'communal_religious', 'xenophobic_regional',
    'misogynistic_sexual', 'abusive_profanity', 'passive_aggressive'
]

# ==========================================
# 2. INFERENCE FUNCTION
# ==========================================
def analyze_hinglish(text, threshold=0.50):
    # Tokenize input text
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=128
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        # Convert raw logits to probabilities (0.0 to 1.0)
        probs = torch.sigmoid(logits).cpu().numpy()[0]

    print(f"\n💬 Text: \"{text}\"")
    print("-" * 55)

    flagged = False
    for cat, prob in zip(categories, probs):
        status = "🚨 FLAGGED" if prob >= threshold else "  OK     "
        print(f"  [{status}] {cat:<22} : {prob*100:5.1f}%")
        if prob >= threshold:
            flagged = True

    if not flagged:
        print("  🟢 Clean sentence (No categories flagged above threshold)")

# ==========================================
# 3. TEST BENCHMARK SENTENCES
# ==========================================
test_samples = [
    "Bihari log idhar aake Bangalore ka culture kharab kar rahe hain",
    "Bhai matches kab shuru hone wale hain, late ho raha hai",
    "Teri toh aisi ki taisi, chup chaap baith ja",
    "Ye sab specific caste ke log hi corruption karte hain"
]

print("🚀 Running Inference Benchmark...")
for sample in test_samples:
    analyze_hinglish(sample, threshold=0.50)

📂 Loading trained V1 MuRIL model...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

🚀 Running Inference Benchmark...

💬 Text: "Bihari log idhar aake Bangalore ka culture kharab kar rahe hain"
-------------------------------------------------------
  [🚨 FLAGGED] casteist               :  92.1%
  [🚨 FLAGGED] communal_religious     :  87.4%
  [🚨 FLAGGED] xenophobic_regional    :  92.2%
  [🚨 FLAGGED] misogynistic_sexual    :  63.4%
  [🚨 FLAGGED] abusive_profanity      :  50.4%
  [🚨 FLAGGED] passive_aggressive     :  52.7%

💬 Text: "Bhai matches kab shuru hone wale hain, late ho raha hai"
-------------------------------------------------------
  [  OK     ] casteist               :   2.1%
  [  OK     ] communal_religious     :   4.0%
  [  OK     ] xenophobic_regional    :   5.1%
  [  OK     ] misogynistic_sexual    :   5.3%
  [  OK     ] abusive_profanity      :   4.7%
  [  OK     ] passive_aggressive     :  32.2%
  🟢 Clean sentence (No categories flagged above threshold)

💬 Text: "Teri toh aisi ki taisi, chup chaap baith ja"
-----------------------------------------------

In [2]:
# @title
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from transformers.trainer_utils import get_last_checkpoint

# ==========================================
# 1. DEFINE PATHS (UPDATED FOR V1.5)
# ==========================================
DRIVE_PROJECT_DIR = '/content/drive/MyDrive/Hinglish_NLP_Project/Production'
DATASET_PATH = os.path.join(DRIVE_PROJECT_DIR, 'muril_training_data_gemini.csv')
CHECKPOINT_DIR = os.path.join(DRIVE_PROJECT_DIR, 'checkpoints/muril_v1.5_weighted')
FINAL_MODEL_DIR = os.path.join(DRIVE_PROJECT_DIR, 'models/muril_v1.5_final')

print("📂 Loading dataset from Drive...")
df = pd.read_csv(DATASET_PATH)

# Clean text column to prevent tokenizer crashes on NaN/numerical values
df = df.dropna(subset=['text'])
df['text'] = df['text'].astype(str)

categories = [
    'casteist', 'communal_religious', 'xenophobic_regional',
    'misogynistic_sexual', 'abusive_profanity', 'passive_aggressive'
]

# ==========================================
# 2. CALCULATE DAMPENED LOSS WEIGHTS (V1.5 FIX)
# ==========================================
num_samples = len(df)
pos_counts = df[categories].sum().values
neg_counts = num_samples - pos_counts

# Apply square root damping (np.sqrt) to stop panic-triggering on rare classes
pos_weights = np.sqrt(neg_counts / (pos_counts + 1e-5))
pos_weight_tensor = torch.tensor(pos_weights, dtype=torch.float32)

print("\n⚖️ V1.5 Dampened Loss Weights (pos_weight):")
for cat, weight in zip(categories, pos_weights):
    print(f"  • {cat:<22}: {weight:.2f}x penalty on error")

# Format labels for Hugging Face multi-label classification
df['labels'] = df[categories].values.tolist()
df['labels'] = df['labels'].apply(lambda x: [float(i) for i in x])

hf_dataset = Dataset.from_pandas(df[['text', 'labels']])
split_dataset = hf_dataset.train_test_split(test_size=0.2, seed=42)

# ==========================================
# 3. INITIALIZE MuRIL & TOKENIZER
# ==========================================
print("\n🧠 Loading MuRIL tokenizer and base model...")
model_id = "google/muril-base-cased"

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForSequenceClassification.from_pretrained(
    model_id,
    num_labels=len(categories),
    problem_type="multi_label_classification",
    id2label={idx: label for idx, label in enumerate(categories)},
    label2id={label: idx for idx, label in enumerate(categories)}
)

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

print("⚙️ Tokenizing datasets...")
tokenized_datasets = split_dataset.map(tokenize_function, batched=True)

# ==========================================
# 4. CUSTOM WEIGHTED TRAINER
# ==========================================
class WeightedMultiLabelTrainer(Trainer):
    def __init__(self, pos_weight=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.pos_weight = pos_weight

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        device = logits.device
        pos_weight = self.pos_weight.to(device) if self.pos_weight is not None else None

        loss_fct = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        loss = loss_fct(logits, labels)

        return (loss, outputs) if return_outputs else loss

# ==========================================
# 5. TRAINING ARGUMENTS & AUTO-RESUME LOGIC
# ==========================================
training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,            # Saves checkpoints to v1.5 folder
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    fp16=True,
)

trainer = WeightedMultiLabelTrainer(
    pos_weight=pos_weight_tensor,
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    processing_class=tokenizer,           # Uses updated processing_class parameter
)

# Search for existing checkpoint in Drive
last_checkpoint = get_last_checkpoint(CHECKPOINT_DIR) if os.path.exists(CHECKPOINT_DIR) else None

if last_checkpoint is not None:
    print(f"\n🔄 Found existing checkpoint at: {last_checkpoint}")
    print("🚀 Resuming training from checkpoint...")
    trainer.train(resume_from_checkpoint=last_checkpoint)
else:
    print("\n🚀 Starting fresh V1.5 training...")
    trainer.train()

# ==========================================
# 6. SAVE FINAL MODEL & TOKENIZER
# ==========================================
print(f"\n💾 Saving final V1.5 model and tokenizer to Drive: {FINAL_MODEL_DIR}")
trainer.save_model(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)

print("\n" + "="*60)
print("🎉 MuRIL V1.5 WEIGHTED TRAINING COMPLETE!")
print(f"📁 Checkpoints Location : {CHECKPOINT_DIR}")
print(f"🚀 Final Model Location : {FINAL_MODEL_DIR}")
print("="*60)

📂 Loading dataset from Drive...

⚖️ V1.5 Dampened Loss Weights (pos_weight):
  • casteist              : 8.89x penalty on error
  • communal_religious    : 2.56x penalty on error
  • xenophobic_regional   : 7.17x penalty on error
  • misogynistic_sexual   : 5.53x penalty on error
  • abusive_profanity     : 2.86x penalty on error
  • passive_aggressive    : 2.68x penalty on error

🧠 Loading MuRIL tokenizer and base model...


config.json:   0%|          | 0.00/411 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/206 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/3.16M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/113 [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


💾 Saving final V1.5 model and tokenizer to Drive: /content/drive/MyDrive/Hinglish_NLP_Project/Production/models/muril_v1.5_final


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


🎉 MuRIL V1.5 WEIGHTED TRAINING COMPLETE!
📁 Checkpoints Location : /content/drive/MyDrive/Hinglish_NLP_Project/Production/checkpoints/muril_v1.5_weighted
🚀 Final Model Location : /content/drive/MyDrive/Hinglish_NLP_Project/Production/models/muril_v1.5_final


In [3]:
# @title
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# ==========================================
# 1. LOAD V1.5 MODEL
# ==========================================
MODEL_PATH = '/content/drive/MyDrive/Hinglish_NLP_Project/Production/models/muril_v1.5_final'

print("📂 Loading V1.5 Model for Spot Check...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

categories = [
    'casteist', 'communal_religious', 'xenophobic_regional',
    'misogynistic_sexual', 'abusive_profanity', 'passive_aggressive'
]

# ==========================================
# 2. THE 60-SENTENCE HINGLISH TEST SUITE
# ==========================================
test_suite = {
    "xenophobic_regional": [
        "Bihari log idhar aake poora city ganda kar rahe hain, wapas bhejo inko.",
        "Ye madrasi log ko na hindi bolni aati hai na dhang se rehna, bas complain karte hain.",
        "Delhi wale bas gundagardi karte hain, civic sense zero hota hai inme.",
        "UP aur Bihar ke log free ka ration khane aate hain humare tax ke paise pe.",
        "North East walo ko dekh ke lagta hi nahi ki ye India se hain.",
        "Mumbai me bahar ke logo ki entry ban karni chahiye, job kha rahe hain.",
        "Bengali log bas baatein banate hain, kaam karna inke bas ka nahi hai.",
        "Gujarati log sirf paise ka sochte hain, desh se koi matlab nahi hai inko.",
        "Haryana wale bina matlab badmash bante hain, akal bilkul nahi hoti.",
        "South walo ka ego bohot zyada hai, national language bolne me inko sharam aati hai."
    ],
    "communal_religious": [
        "Inke religion me toh bas yahi sikhaya jaata hai, dusro se nafrat karna.",
        "Ye log desh ke law ko nahi maante, sirf apne religious rules follow karte hain.",
        "Festival ke naam pe road block karke sabko pareshan karna inka routine hai.",
        "Inki population aise hi badhti rahi toh country ka culture khatam ho jayega.",
        "Apne festival pe noise pollution dikhta nahi, humare time pe supreme court yaad aata hai.",
        "Ye community wale kabhi loyal nahi ho sakte, history utha ke dekh lo.",
        "Inke area me jao toh lagta hai kisi aur hi country me aa gaye hain.",
        "Sab fundamentalists ek hi community se kyu nikalte hain, socho zara.",
        "Ye log sacche indians nahi hain, inka loyalty bahar ke mulk ke sath hai.",
        "Charity ke naam pe conversion chal raha hai inka poore state me."
    ],
    "casteist": [
        "Reservation ke bina inko clerk ki job bhi na mile, merit toh zero hai inme.",
        "Ye log quota se admit ho jate hain aur fir college ka standard gira dete hain.",
        "Inki caste walo ka yahi kaam hai, freebies pe jeena aur complain karna.",
        "Upper caste walo ne hi desh ko aage badhaya hai, baaki sab bas faida utha rahe hain.",
        "Kitna bhi padh le, inki soch aur behaviour wahi neech level ka hi rahega.",
        "General category walo ka tax khake ye log free ki degree le rahe hain.",
        "Inko equal status de diya isliye ab humare sar pe chadh ke naach rahe hain.",
        "Specific jaati ke log hi humesha aisi cheap harkatein karte dikhenge.",
        "Inka surname dekh ke hi samajh aa gaya tha ki inka standard kya hoga.",
        "Sarkari office me saare quota wale bhare hain, isliye koi kaam dhang se nahi hota."
    ],
    "misogynistic_sexual": [
        "Ladakiya games kyu khelti hain bro, tum se na ho payega kitchen me jao.",
        "Iska promotion kaam se nahi mila hoga, sabko pata hai kaise milta hai.",
        "Aaj kal ki ladkiyo ko bas rich ladka chahiye ATM ki tarah use karne ke liye.",
        "Aise kapde pehen ke live stream karegi toh log aisi hi comments karenge na.",
        "Females me driving sense aur gaming logic by default zero hota hai.",
        "Ye sab attention seek karne ka tarika hai, real life me koi nahi puchta isko.",
        "Ladki hai isliye simp log bhar bhar ke views de rahe hain, gameplay toh third class hai.",
        "Divorce leke alimony ke paise pe mauj kar rahi hai, yahi business ban gaya hai ab.",
        "Char logo ke samne bolne aati ho, pehle dhang se rehna seekh lo.",
        "Women rights ke naam pe bas men ko harass karna aata hai inko."
    ],
    "abusive_profanity": [
        "Abe nallhe, tu kyu beech me apni bakwaas kar raha hai, nikal yaha se.",
        "Noob sala, game khelna nahi aata toh install kyu kiya bc.",
        "Teri toh aisi ki taisi, chup chaap apna kaam kar warna muh tod dunga.",
        "Kaisa gadha admin hai, basic sense nahi hai isko server manage karne ka.",
        "Abe chomu, dimag ghutne me hai kya tera? Kitni baar bolu ek hi baat.",
        "Saale haramkhor, meri rank drop karwa di teri wajah se game har gaye.",
        "Bhai tu dimaag se paidal hai kya, itna bekar log aaj tak nahi dekha mc.",
        "Apna gyan apne paas rakh bkl, humko mat sikha kya karna hai.",
        "Ek number ka fraud banda hai tu, sharam bachi hai ya wo bhi bech khayi?",
        "Aukad me reh apni, zyada hero banne ki koshish mat kar idhar."
    ],
    "passive_aggressive": [
        "Bhai tumhara basic knowledge dekh ke lagta hai tumhe KG se wapas start karna chahiye.",
        "Wau, kitna great logical argument diya hai tumne, taaliya ho jaye inke liye.",
        "Aap jaise smart log agar duniya me honge toh rocket science ki zaroorat hi nahi padegi.",
        "Don't worry bro, har kisi ke bas ki baat nahi hoti simple instructions samajhna.",
        "Achha hua tumne bata diya, warna hum sab toh yaha bewakoof hi baithe the.",
        "Tumse bohot umeed thi mujhe, par tumne prove kar diya ki main galat tha.",
        "Sirf angrezi bolne se koi intelligent nahi ban jata, thoda common sense bhi chahiye.",
        "Aapka effort bohot cute hai, par agli baar kisi samajhdar insaan ko kaam karne dena.",
        "Koi baat nahi, galti sabse hoti hai, bas tumse thodi zyada hi hoti hai.",
        "Lagta hai aap bohot busy the isliye bina soche samjhe ye reply type kar diya."
    ]
}

# ==========================================
# 3. RUN THE SPOT CHECK AUDIT
# ==========================================
print("🚀 Running 60-Sentence Boundary & Label Bleed Audit...\n")

bleed_errors = 0
correct_primary_hits = 0

for target_category, sentences in test_suite.items():
    print(f"==================================================")
    print(f"📌 TESTING CATEGORY: {target_category.upper()}")
    print(f"==================================================")

    for idx, text in enumerate(sentences, 1):
        inputs = tokenizer(text, return_tensors="pt", truncation=True, padding="max_length", max_length=128).to(device)
        with torch.no_grad():
            probs = torch.sigmoid(model(**inputs).logits).cpu().numpy()[0]

        flagged = [cat for cat, prob in zip(categories, probs) if prob >= 0.50]

        # Check diagnostics
        hit_target = target_category in flagged
        if hit_target:
            correct_primary_hits += 1

        # An error is if it triggered 3+ alarms (label bleed) or missed the primary category
        is_bleeding = len(flagged) >= 3
        if is_bleeding:
            bleed_errors += 1

        status_icon = "🚨 BLEED" if is_bleeding else ("✅ OK   " if hit_target else "❌ MISSED")
        print(f"  [{status_icon}] Q{idx:02d}: \"{text[:45]}...\" -> Flagged: {flagged if flagged else ['NONE']}")
    print()

print("📊 FINAL AUDIT SUMMARY")
print("-" * 55)
print(f"  • Target Category Recall : {correct_primary_hits}/60 ({(correct_primary_hits/60)*100:.1f}%)")
print(f"  • Heavy Bleed (≥3 tags)  : {bleed_errors}/60 ({(bleed_errors/60)*100:.1f}%)")
if bleed_errors <= 6:
    print("  🟢 PASSED: Label bleed is well under control (<10% error rate).")
else:
    print("  🟡 WARNING: Some label bleed persists. Review which categories over-trigger.")

📂 Loading V1.5 Model for Spot Check...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

🚀 Running 60-Sentence Boundary & Label Bleed Audit...

📌 TESTING CATEGORY: XENOPHOBIC_REGIONAL
  [🚨 BLEED] Q01: "Bihari log idhar aake poora city ganda kar ra..." -> Flagged: ['casteist', 'communal_religious', 'xenophobic_regional', 'misogynistic_sexual', 'abusive_profanity', 'passive_aggressive']
  [🚨 BLEED] Q02: "Ye madrasi log ko na hindi bolni aati hai na ..." -> Flagged: ['casteist', 'communal_religious', 'xenophobic_regional', 'misogynistic_sexual', 'abusive_profanity', 'passive_aggressive']
  [❌ MISSED] Q03: "Delhi wale bas gundagardi karte hain, civic s..." -> Flagged: ['communal_religious']
  [🚨 BLEED] Q04: "UP aur Bihar ke log free ka ration khane aate..." -> Flagged: ['casteist', 'communal_religious', 'xenophobic_regional', 'misogynistic_sexual', 'abusive_profanity', 'passive_aggressive']
  [❌ MISSED] Q05: "North East walo ko dekh ke lagta hi nahi ki y..." -> Flagged: ['NONE']
  [❌ MISSED] Q06: "Mumbai me bahar ke logo ki entry ban karni ch..." -> Flagged: ['NONE']
  [🚨 BLEE

In [4]:
# @title
import os
import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import f1_score, precision_score, recall_score

# ==========================================
# 1. LOAD MODEL & UNSEEN VALIDATION DATA
# ==========================================
DRIVE_PROJECT_DIR = '/content/drive/MyDrive/Hinglish_NLP_Project/Production'
MODEL_PATH = os.path.join(DRIVE_PROJECT_DIR, 'models/muril_v1.5_final')
DATASET_PATH = os.path.join(DRIVE_PROJECT_DIR, 'muril_training_data_gemini.csv')

print("📂 Loading V1.5 Model and Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

categories = [
    'casteist', 'communal_religious', 'xenophobic_regional',
    'misogynistic_sexual', 'abusive_profanity', 'passive_aggressive'
]

print("📂 Recreating unseen 20% validation split...")
df = pd.read_csv(DATASET_PATH).dropna(subset=['text'])
df['text'] = df['text'].astype(str)
_, test_df = np.split(df.sample(frac=1, random_state=42), [int(.8 * len(df))])

# Use 1,000 validation rows for a blindingly fast threshold sweep
eval_df = test_df.head(1000).copy()

# ==========================================
# 2. RUN BATCH INFERENCE TO GET RAW PROBABILITIES
# ==========================================
print(f"🚀 Extracting raw probability logits across {len(eval_df)} unseen samples...")
inputs = tokenizer(
    eval_df['text'].tolist(), padding="max_length", truncation=True, max_length=128, return_tensors="pt"
)

dataset = TensorDataset(inputs['input_ids'], inputs['attention_mask'])
dataloader = DataLoader(dataset, batch_size=32)

all_probs = []
with torch.no_grad():
    for batch in dataloader:
        input_ids, attention_mask = [b.to(device) for b in batch]
        logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
        probs = torch.sigmoid(logits).cpu().numpy()
        all_probs.append(probs)

y_probs = np.vstack(all_probs)
y_true = np.array(eval_df[categories].values.tolist(), dtype=int)

# ==========================================
# 3. SWEEP FOR OPTIMAL THRESHOLD PER CLASS
# ==========================================
print("\n🔍 OPTIMAL THRESHOLD SWEEP (Maximizing F1-Score per Category)")
print("=" * 75)
print(f"  {'Category':<22} | {'Old (0.50)':<10} -> {'New Best':<10} | {'Precision':<10} | {'Recall':<8}")
print("-" * 75)

optimal_thresholds = {}

for idx, cat in enumerate(categories):
    best_thresh = 0.50
    best_f1 = 0.0
    best_p = 0.0
    best_r = 0.0

    # Sweep thresholds from 0.10 to 0.90 in steps of 0.02
    for thresh in np.arange(0.10, 0.92, 0.02):
        preds = (y_probs[:, idx] >= thresh).astype(int)
        f1 = f1_score(y_true[:, idx], preds, zero_division=0)

        if f1 > best_f1:
            best_f1 = f1
            best_thresh = thresh
            best_p = precision_score(y_true[:, idx], preds, zero_division=0)
            best_r = recall_score(y_true[:, idx], preds, zero_division=0)

    optimal_thresholds[cat] = round(float(best_thresh), 2)
    old_f1 = f1_score(y_true[:, idx], (y_probs[:, idx] >= 0.50).astype(int), zero_division=0)

    print(f"  • {cat:<20} | F1: {old_f1:.2f}    -> Thresh: {best_thresh:.2f} | P: {best_p:.2f}   | R: {best_r:.2f}")

print("\n🚀 YOUR PRODUCTION THRESHOLD DICTIONARY (Copy & Paste into Inference):")
print("-" * 75)
print(f"custom_thresholds = {optimal_thresholds}")

📂 Loading V1.5 Model and Tokenizer...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

📂 Recreating unseen 20% validation split...


/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


🚀 Extracting raw probability logits across 1000 unseen samples...

🔍 OPTIMAL THRESHOLD SWEEP (Maximizing F1-Score per Category)
  Category               | Old (0.50) -> New Best   | Precision  | Recall  
---------------------------------------------------------------------------
  • casteist             | F1: 0.41    -> Thresh: 0.78 | P: 1.00   | R: 0.64
  • communal_religious   | F1: 0.57    -> Thresh: 0.48 | P: 0.44   | R: 0.79
  • xenophobic_regional  | F1: 0.05    -> Thresh: 0.72 | P: 0.50   | R: 0.06
  • misogynistic_sexual  | F1: 0.44    -> Thresh: 0.46 | P: 0.43   | R: 0.48
  • abusive_profanity    | F1: 0.52    -> Thresh: 0.58 | P: 0.58   | R: 0.58
  • passive_aggressive   | F1: 0.14    -> Thresh: 0.22 | P: 0.27   | R: 0.71

🚀 YOUR PRODUCTION THRESHOLD DICTIONARY (Copy & Paste into Inference):
---------------------------------------------------------------------------
custom_thresholds = {'casteist': 0.78, 'communal_religious': 0.48, 'xenophobic_regional': 0.72, 'misogynistic_s

In [5]:
# @title
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_PATH = '/content/drive/MyDrive/Hinglish_NLP_Project/Production/models/muril_v1.5_final'

print("📂 Loading V1.5 Model with Custom Decision Thresholds...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

categories = [
    'casteist', 'communal_religious', 'xenophobic_regional',
    'misogynistic_sexual', 'abusive_profanity', 'passive_aggressive'
]

# 🚀 YOUR OPTIMIZED PRODUCTION THRESHOLDS
custom_thresholds = {
    'casteist': 0.78,
    'communal_religious': 0.48,
    'xenophobic_regional': 0.72,
    'misogynistic_sexual': 0.46,
    'abusive_profanity': 0.58,
    'passive_aggressive': 0.22
}

test_suite = {
    "casteist": [
        "Reservation ke bina inko clerk ki job bhi na mile, merit toh zero hai inme.",
        "Upper caste walo ne hi desh ko aage badhaya hai, baaki sab bas faida utha rahe hain.",
        "Specific jaati ke log hi humesha aisi cheap harkatein karte dikhenge."
    ],
    "passive_aggressive": [
        "Wau, kitna great logical argument diya hai tumne, taaliya ho jaye inke liye.",
        "Aap jaise smart log agar duniya me honge toh rocket science ki zaroorat hi nahi padegi.",
        "Sirf angrezi bolne se koi intelligent nahi ban jata, thoda common sense bhi chahiye."
    ],
    "abusive_profanity": [
        "Abe nallhe, tu kyu beech me apni bakwaas kar raha hai, nikal yaha se.",
        "Teri toh aisi ki taisi, chup chaap apna kaam kar warna muh tod dunga.",
        "Saale haramkhor, meri rank drop karwa di teri wajah se game har gaye."
    ]
}

print("\n🚀 Running Spot Check with Custom Thresholds...\n")

for target_category, sentences in test_suite.items():
    print(f"📌 TESTING CATEGORY: {target_category.upper()} (Threshold: {custom_thresholds[target_category]})")
    print("-" * 65)
    for idx, text in enumerate(sentences, 1):
        inputs = tokenizer(text, return_tensors="pt", truncation=True, padding="max_length", max_length=128).to(device)
        with torch.no_grad():
            probs = torch.sigmoid(model(**inputs).logits).cpu().numpy()[0]

        # Dynamically evaluate against custom thresholds instead of flat 0.50
        flagged = [cat for cat, prob in zip(categories, probs) if prob >= custom_thresholds[cat]]

        is_bleeding = len(flagged) >= 3
        hit_target = target_category in flagged
        status_icon = "🚨 BLEED" if is_bleeding else ("✅ OK   " if hit_target else "❌ MISSED")

        print(f"  [{status_icon}] \"{text[:45]}...\" -> {flagged if flagged else ['NONE']}")
    print()

📂 Loading V1.5 Model with Custom Decision Thresholds...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


🚀 Running Spot Check with Custom Thresholds...

📌 TESTING CATEGORY: CASTEIST (Threshold: 0.78)
-----------------------------------------------------------------
  [❌ MISSED] "Reservation ke bina inko clerk ki job bhi na ..." -> ['misogynistic_sexual', 'passive_aggressive']
  [🚨 BLEED] "Upper caste walo ne hi desh ko aage badhaya h..." -> ['casteist', 'communal_religious', 'misogynistic_sexual', 'passive_aggressive']
  [🚨 BLEED] "Specific jaati ke log hi humesha aisi cheap h..." -> ['casteist', 'communal_religious', 'misogynistic_sexual', 'passive_aggressive']

📌 TESTING CATEGORY: PASSIVE_AGGRESSIVE (Threshold: 0.22)
-----------------------------------------------------------------
  [✅ OK   ] "Wau, kitna great logical argument diya hai tu..." -> ['passive_aggressive']
  [❌ MISSED] "Aap jaise smart log agar duniya me honge toh ..." -> ['NONE']
  [✅ OK   ] "Sirf angrezi bolne se koi intelligent nahi ba..." -> ['passive_aggressive']

📌 TESTING CATEGORY: ABUSIVE_PROFANITY (Threshold: 0.58